In [1]:
import numpy as np
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import pandas as pd
import glob 
import os
import pickle
from pathlib import Path
import time
import pandas as pd

import scipy.signal as signal
import scipy.fft as fft
import scipy.stats as stats

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import glob 
import pywt #PyWavelets package
import matplotlib.colors as colors
from matplotlib.colors import LogNorm



from data_process import select_df,read_and_process_data ,fac_decompose_dB

In [2]:
with open(Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice-location-ready/"+"cassini_enis_2004_2017_60s.pkl"), "rb") as f:
    cassini_location = pickle.load(f)

In [3]:
def merge_ICW_with_location(labels_df, cassini_location, tolerance='5min'):
    labels = labels_df.copy()
    loc = cassini_location.copy()

    labels['interval_start'] = pd.to_datetime(labels['interval_start']).dt.tz_localize(None)
    labels['interval_end'] = pd.to_datetime(labels['interval_end']).dt.tz_localize(None)
    loc['utc'] = pd.to_datetime(loc['utc']).dt.tz_localize(None)

    # Use interval midpoint as representative time
    labels['time'] = labels['interval_start'] + (
        labels['interval_end'] - labels['interval_start']
    ) / 2

    labels = labels.sort_values('time')
    loc = loc.sort_values('utc')

    # keep_cols = [
    #     'utc', 'x_ksm_Rs', 'y_ksm_Rs', 'z_ksm_Rs',
    #     'r_Rs', 'lat_deg', 'lon_deg'
    # ]
    # keep_cols = [c for c in keep_cols if c in loc.columns]

    merged = pd.merge_asof(
        labels,
        loc,
        left_on='time',
        right_on='utc',
        direction='nearest',
        tolerance=pd.Timedelta(tolerance)
    )

    # return merged.dropna(subset=['r_Rs']).reset_index(drop=True)
    return merged.reset_index(drop=True)
    

# prep

## 1D

In [ ]:
def one_d_occurrence_statistics(
    df,
    x_col,
    flag_col='is_ICW',
    bins=None,
    x_range=None,
    min_samples=1
):
    """
    Generic 1D occurrence statistics.

    Occurrence fraction = number of flagged intervals / total intervals in each bin.

    Returns columns:
        bin
        n_total
        n_wave
        x_mean
        occurrence_fraction
        occurrence_percent
    """

    d = df.dropna(subset=[x_col, flag_col]).copy()
    d[flag_col] = d[flag_col].astype(bool)

    if x_range is not None:
        d = d[
            (d[x_col] >= x_range[0]) &
            (d[x_col] <= x_range[1])
        ].copy()

    if bins is None:
        bins = np.linspace(d[x_col].min(), d[x_col].max(), 50)

    d['bin'] = pd.cut(d[x_col], bins=bins, include_lowest=True)

    stat = d.groupby('bin', observed=True).agg(
        n_total=(flag_col, 'size'),
        n_wave=(flag_col, 'sum'),
        x_mean=(x_col, 'mean')
    ).reset_index()

    stat['occurrence_fraction'] = stat['n_wave'] / stat['n_total']
    stat['occurrence_percent'] = 100 * stat['occurrence_fraction']

    stat.loc[
        stat['n_total'] < min_samples,
        ['occurrence_fraction', 'occurrence_percent']
    ] = np.nan

    return stat

In [ ]:
def plot_one_d_occurrence(
    stat,
    x_label,
    title,
    y_mode='fraction',
    xlim=None,
    xticks=None,
    ax=None,
    marker='o'
):
    """
    Plot 1D occurrence rate from one_d_occurrence_statistics().
    """

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.figure

    if y_mode == 'percent':
        y_col = 'occurrence_percent'
        y_label = 'ICW Occurrence Rate (%)'
    else:
        y_col = 'occurrence_fraction'
        y_label = 'ICW Fraction'

    ax.plot(
        stat['x_mean'],
        stat[y_col],
        marker=marker
    )

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)

    if xlim is not None:
        ax.set_xlim(*xlim)

    if xticks is not None:
        ax.set_xticks(xticks)

    ax.grid(True, alpha=0.3)

    return fig, ax

In [ ]:
def one_d_sample_statistics(
    df,
    x_col,
    bins=None,
    x_range=None,
    min_samples=1
):
    """
    Generic 1D sample-number statistics.

    Returns columns:
        bin
        n_total
        x_mean
    """

    d = df.dropna(subset=[x_col]).copy()

    if x_range is not None:
        d = d[
            (d[x_col] >= x_range[0]) &
            (d[x_col] <= x_range[1])
        ].copy()

    if bins is None:
        bins = np.linspace(d[x_col].min(), d[x_col].max(), 50)

    d['bin'] = pd.cut(d[x_col], bins=bins, include_lowest=True)

    stat = d.groupby('bin', observed=True).agg(
        n_total=(x_col, 'size'),
        x_mean=(x_col, 'mean')
    ).reset_index()

    stat.loc[stat['n_total'] < min_samples, 'n_total'] = np.nan

    return stat



In [ ]:
def plot_one_d_samples(
    stat,
    x_label,
    title,
    xlim=None,
    xticks=None,
    ax=None,
    marker='o',
    log_y=False
):
    """
    Plot 1D sample numbers from one_d_sample_statistics().
    """

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.figure

    ax.plot(
        stat['x_mean'],
        stat['n_total'],
        marker=marker
    )

    ax.set_xlabel(x_label)
    ax.set_ylabel('Number of Samples')
    ax.set_title(title)

    if log_y:
        ax.set_yscale('log')

    if xlim is not None:
        ax.set_xlim(*xlim)

    if xticks is not None:
        ax.set_xticks(xticks)

    ax.grid(True, alpha=0.3)

    return fig, ax

## 2D

In [ ]:
def make_wave_occurrence_maps(
    df,
    L_col='L_approx',
    MLT_col='MLT_hr',
    MLAT_col='MLAT_approx_deg',
    flag_col='',
    L_range=(2, 20),
    MLAT_range=(-60, 60),
    dL=0.25,
    dMLT=0.5,
    dMLAT=2.5,
    min_samples=10
):
    """
    Make Long et al.-style occurrence maps:
        L-MLT:  dL x dMLT
        L-MLAT: dL x dMLAT

    Returns a dictionary containing occurrence maps and sample-number maps.
    """

    d = df.copy()

    needed = [L_col, MLT_col, MLAT_col, flag_col]
    missing = [c for c in needed if c not in d.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    d = d[needed].replace([np.inf, -np.inf], np.nan).dropna()
    d[flag_col] = d[flag_col].astype(bool)

    # Clean ranges
    d = d[
        (d[L_col] >= L_range[0]) &
        (d[L_col] <= L_range[1]) &
        (d[MLT_col] >= 0) &
        (d[MLT_col] < 24) &
        (d[MLAT_col] >= MLAT_range[0]) &
        (d[MLAT_col] <= MLAT_range[1])
    ].copy()

    L_edges = np.arange(L_range[0], L_range[1] + dL, dL)
    MLT_edges = np.arange(0, 24 + dMLT, dMLT)
    MLAT_edges = np.arange(MLAT_range[0], MLAT_range[1] + dMLAT, dMLAT)

    # -----------------------
    # L-MLT map
    # -----------------------
    total_L_MLT, _, _ = np.histogram2d(
        d[L_col],
        d[MLT_col],
        bins=[L_edges, MLT_edges]
    )

    wave_L_MLT, _, _ = np.histogram2d(
        d.loc[d[flag_col], L_col],
        d.loc[d[flag_col], MLT_col],
        bins=[L_edges, MLT_edges]
    )

    occ_L_MLT = 100.0 * wave_L_MLT / total_L_MLT
    occ_L_MLT[total_L_MLT < min_samples] = np.nan

    # -----------------------
    # L-MLAT map
    # -----------------------
    total_L_MLAT, _, _ = np.histogram2d(
        d[L_col],
        d[MLAT_col],
        bins=[L_edges, MLAT_edges]
    )

    wave_L_MLAT, _, _ = np.histogram2d(
        d.loc[d[flag_col], L_col],
        d.loc[d[flag_col], MLAT_col],
        bins=[L_edges, MLAT_edges]
    )

    occ_L_MLAT = 100.0 * wave_L_MLAT / total_L_MLAT
    occ_L_MLAT[total_L_MLAT < min_samples] = np.nan

    return {
        'L_edges': L_edges,
        'MLT_edges': MLT_edges,
        'MLAT_edges': MLAT_edges,

        'occ_L_MLT': occ_L_MLT,
        'Ns_L_MLT': total_L_MLT,

        'occ_L_MLAT': occ_L_MLAT,
        'Ns_L_MLAT': total_L_MLAT,

        'clean_df': d
    }

In [ ]:
def plot_L_MLT_occurrence(maps, title='ICW Occurrence Rate in L-MLT'):
    L_edges = maps['L_edges']
    MLT_edges = maps['MLT_edges']
    occ = maps['occ_L_MLT']

    theta_edges = 2 * np.pi * MLT_edges / 24.0

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})

    pcm = ax.pcolormesh(
        theta_edges,
        L_edges,
        occ,
        shading='auto',
        vmin=0,
        vmax=100
    )

    ax.set_theta_zero_location('E')   # 00 MLT on right
    ax.set_theta_direction(1)         # 06 top, 12 left, 18 bottom

    ax.set_xticks(2 * np.pi * np.array([0, 6, 12, 18]) / 24)
    ax.set_xticklabels(['00', '06', '12', '18'])

    ax.set_ylabel('L shell')
    ax.set_title(title)

    # Enceladus and Dione reference circles
    theta = np.linspace(0, 2 * np.pi, 500)
    ax.plot(theta, np.full_like(theta, 4.0), c='C3',linestyle='--', linewidth=1.5, label='Enceladus L~4')
    ax.plot(theta, np.full_like(theta, 6.2), c='C1',linestyle='--', linewidth=1.5, label='Dione L~6.2')

    cbar = fig.colorbar(pcm, ax=ax, pad=0.1)
    cbar.set_label('Occurrence Rate (%)')

    ax.legend(loc='upper right', bbox_to_anchor=(1.85, 1.15))

    plt.show()

In [ ]:
def plot_L_MLAT_occurrence(
    maps,
    title='ICW Occurrence Rate in L-MLAT',
    vmin=0,
    vmax=100
):
    L_edges = maps['L_edges']
    MLAT_edges = maps['MLAT_edges']
    occ = maps['occ_L_MLAT']

    theta_edges = np.radians(MLAT_edges)

    fig, ax = plt.subplots(
        figsize=(7, 7),
        subplot_kw={'projection': 'polar'}
    )


    
    pcm = ax.pcolormesh(
        theta_edges,
        L_edges,
        occ,
        shading='auto',
        vmin=vmin,
        vmax=vmax
    )

    ax.set_thetamin(-90)
    ax.set_thetamax(90)

    # ax.set_theta_zero_location('W')
    # ax.set_theta_direction(-1)

    mlat_ticks = np.array([-60, -40, -20, 0, 20, 40, 60])
    theta_ticks = np.radians(mlat_ticks)

    ax.set_xticks(theta_ticks)
    ax.set_xticklabels([f'{x:g}°' for x in mlat_ticks])

    ax.set_xlabel('MLAT')
    ax.set_ylabel('L shell')
    ax.set_title(title)

    theta = np.linspace(-0.5*np.pi, 0.5*np.pi, 400)
    ax.plot(theta, np.full_like(theta, 4.0), linestyle='--', linewidth=1.5, label='Enceladus L~4')
    ax.plot(theta, np.full_like(theta, 6.2), linestyle='--', linewidth=1.5, label='Dione L~6.2')

    ax.grid(True, alpha=0.3)

    cbar = fig.colorbar(pcm, ax=ax, pad=0.1)
    cbar.set_label('Occurrence Rate (%)')

    ax.legend(loc='upper right', bbox_to_anchor=(1.85, 1.1))

    plt.show()

In [ ]:
def plot_L_MLT_samples(
    maps,
    title='Number of Samples in L-MLT',
    vmin=10,
    vmax=None
):
    L_edges = maps['L_edges']
    MLT_edges = maps['MLT_edges']
    Ns = maps['Ns_L_MLT'].copy()

    Ns[Ns < 1] = np.nan

    if vmax is None:
        vmax = np.nanmax(Ns)

    theta_edges = 2 * np.pi * MLT_edges / 24.0

    fig, ax = plt.subplots(
        figsize=(7, 7),
        subplot_kw={'projection': 'polar'}
    )

    pcm = ax.pcolormesh(
        theta_edges,
        L_edges,
        Ns,
        shading='auto',
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )

    ax.set_theta_zero_location('E')   # 00 MLT on right
    ax.set_theta_direction(1)         # 06 top, 12 left, 18 bottom

    ax.set_xticks(2 * np.pi * np.array([0, 6, 12, 18]) / 24)
    ax.set_xticklabels(['00', '06', '12', '18'])

    ax.set_ylabel('L shell')
    ax.set_title(title)

    theta = np.linspace(0, 2 * np.pi, 500)
    ax.plot(theta, np.full_like(theta, 4.0), linestyle='--', linewidth=1.5, label='Enceladus L~4')
    ax.plot(theta, np.full_like(theta, 6.2), linestyle='--', linewidth=1.5, label='Dione L~6.2')

    cbar = fig.colorbar(pcm, ax=ax, pad=0.1)
    cbar.set_label('Number of Samples')

    ax.legend(loc='upper right', bbox_to_anchor=(1.85, 1.15))

    plt.show()

In [ ]:
def plot_L_MLAT_samples(
    maps,
    title='Number of Samples in L-MLAT',
    vmin=10,
    vmax=None
):
    L_edges = maps['L_edges']
    MLAT_edges = maps['MLAT_edges']
    Ns = maps['Ns_L_MLAT'].copy()

    Ns[Ns < 1] = np.nan

    if vmax is None:
        vmax = np.nanmax(Ns)

    theta_edges = np.radians(MLAT_edges)

    fig, ax = plt.subplots(
        figsize=(7, 7),
        subplot_kw={'projection': 'polar'}
    )

    pcm = ax.pcolormesh(
        theta_edges,
        L_edges,
        Ns,
        shading='auto',
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )

    ax.set_thetamin(-90)
    ax.set_thetamax(90)

    mlat_ticks = np.array([-60, -40, -20, 0, 20, 40, 60])
    theta_ticks = np.radians(mlat_ticks)

    ax.set_xticks(theta_ticks)
    ax.set_xticklabels([f'{x:g}°' for x in mlat_ticks])

    ax.set_xlabel('MLAT')
    ax.set_ylabel('L shell')
    ax.set_title(title)

    theta = np.linspace(-0.5 * np.pi, 0.5 * np.pi, 400)
    ax.plot(theta, np.full_like(theta, 4.0), linestyle='--', linewidth=1.5, label='Enceladus L~4')
    ax.plot(theta, np.full_like(theta, 6.2), linestyle='--', linewidth=1.5, label='Dione L~6.2')

    ax.grid(True, alpha=0.3)

    cbar = fig.colorbar(pcm, ax=ax, pad=0.1)
    cbar.set_label('Number of Samples')

    ax.legend(loc='upper right', bbox_to_anchor=(1.85, 1.1))

    plt.show()

## 3D

In [ ]:
def make_3d_occurrence_table(
    df,
    flag_col='is_ICW',
    L_col='L_approx',
    MLT_col='MLT_hr',
    MLAT_col='MLAT_approx_deg',
    L_range=(2, 20),
    MLAT_range=(-60, 60),
    dL=0.25,
    dMLT=0.5,
    dMLAT=2.5,
    min_samples=10
):
    d = df[[L_col, MLT_col, MLAT_col, flag_col]].copy()
    d = d.replace([np.inf, -np.inf], np.nan).dropna()

    d[flag_col] = d[flag_col].astype(bool)

    d = d[
        (d[L_col] >= L_range[0]) &
        (d[L_col] <= L_range[1]) &
        (d[MLT_col] >= 0) &
        (d[MLT_col] < 24) &
        (d[MLAT_col] >= MLAT_range[0]) &
        (d[MLAT_col] <= MLAT_range[1])
    ].copy()

    L_edges = np.arange(L_range[0], L_range[1] + dL, dL)
    MLT_edges = np.arange(0, 24 + dMLT, dMLT)
    MLAT_edges = np.arange(MLAT_range[0], MLAT_range[1] + dMLAT, dMLAT)

    d['L_bin'] = pd.cut(d[L_col], bins=L_edges, include_lowest=True)
    d['MLT_bin'] = pd.cut(d[MLT_col], bins=MLT_edges, include_lowest=True)
    d['MLAT_bin'] = pd.cut(d[MLAT_col], bins=MLAT_edges, include_lowest=True)

    stat = d.groupby(
        ['L_bin', 'MLT_bin', 'MLAT_bin'],
        observed=True
    ).agg(
        n_total=(flag_col, 'size'),
        n_wave=(flag_col, 'sum'),
        L_mean=(L_col, 'mean'),
        MLT_mean=(MLT_col, 'mean'),
        MLAT_mean=(MLAT_col, 'mean')
    ).reset_index()

    stat['occurrence_fraction'] = stat['n_wave'] / stat['n_total']
    stat['occurrence_percent'] = 100 * stat['occurrence_fraction']

    stat = stat[stat['n_total'] >= min_samples].copy()

    theta = 2 * np.pi * stat['MLT_mean'] / 24.0
    mlat_rad = np.radians(stat['MLAT_mean'])

    # Simple magnetospheric 3D display coordinates
    stat['x'] = stat['L_mean'] * np.cos(theta)
    stat['y'] = stat['L_mean'] * np.sin(theta)
    stat['z'] = stat['L_mean'] * np.sin(mlat_rad)

    return stat

In [ ]:
def plot_3d_occurrence_plotly_go(
    stat3d,
    color_col='occurrence_percent',
    title='3D ICW Occurrence Rate',
    color_range=(0, 100),
    marker_size=6,
    colorscale='Viridis',
    opacity=0.85
):
    """
    3D occurrence-rate plot using plotly.graph_objects.

    Marker size is fixed.
    Marker color shows occurrence rate.

    Required columns in stat3d:
        x, y, z
        L_mean, MLT_mean, MLAT_mean
        n_total, n_wave
        occurrence_percent
    """

    d = stat3d.copy()

    d = d.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['x', 'y', 'z', color_col]
    )

    hover_text = [
        (
            f"L = {L:.2f}<br>"
            f"MLT = {MLT:.2f} hr<br>"
            f"MLAT = {MLAT:.1f}°<br>"
            f"N total = {Nt}<br>"
            f"N wave = {Nw}<br>"
            f"Occurrence = {Occ:.1f}%"
        )
        for L, MLT, MLAT, Nt, Nw, Occ in zip(
            d['L_mean'],
            d['MLT_mean'],
            d['MLAT_mean'],
            d['n_total'],
            d['n_wave'],
            d[color_col]
        )
    ]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=d['x'],
            y=d['y'],
            z=d['z'],
            mode='markers',
            text=hover_text,
            hoverinfo='text',
            marker=dict(
                size=marker_size,
                color=d[color_col],
                colorscale=colorscale,
                cmin=color_range[0],
                cmax=color_range[1],
                opacity=opacity,
                colorbar=dict(
                    title='Occurrence Rate (%)'
                )
            ),
            name='Occurrence'
        )
    )

    fig.update_layout(
        title=title,
        width=1500,
        height=600,
        scene=dict(
            xaxis=dict(title='X-like coordinate'),
            yaxis=dict(title='Y-like coordinate'),
            zaxis=dict(title='Z-like coordinate'),
            aspectmode='data'
        )
    )

    return fig

In [ ]:
def add_L_shell_ring_go(
    fig,
    L=4.0,
    name='Enceladus L~4',
    z=0,
    line_width=5
):
    theta = np.linspace(0, 2 * np.pi, 500)

    x = L * np.cos(theta)
    y = L * np.sin(theta)
    zz = np.full_like(theta, z)

    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=zz,
            mode='lines',
            name=name,
            line=dict(
                width=line_width,
                dash='dash'
            )
        )
    )

    return fig

# load in events

In [4]:
with open('data/check_wave/ICW_mirror_labels_2004_2017.pkl', 'rb') as f:
    labels_df = pickle.load(f)

In [5]:
ICW_loc_df = merge_ICW_with_location(labels_df, cassini_location)

In [6]:
ICW_loc_df

,interval_start,interval_end,is_ICW,is_mirror,label,file_name,B0_nT,fc_Hz_H2Oplus,icw_f_peak_Hz,icw_f_peak_over_fc,...,saturn_z_enis_km,saturn_x_enis_RE,saturn_y_enis_RE,saturn_z_enis_RE,sun_x_enis_km,sun_y_enis_km,sun_z_enis_km,sun_x_enis_RE,sun_y_enis_RE,sun_z_enis_RE
0,2004-07-01 00:10:00.500,2004-07-01 00:20:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,664.963930,0.567292,0.374376,0.659935,...,-1.089610e-12,6.912705e-14,943.626275,-4.323963e-15,1.200089e+09,-2.742972e+08,-5.605835e+08,4.762384e+06,-1.088510e+06,-2.224597e+06
1,2004-07-01 00:20:00.500,2004-07-01 00:30:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,777.296952,0.663126,0.374376,0.564563,...,-1.123629e-12,1.049396e-13,943.762162,-4.458964e-15,1.190720e+09,-3.124353e+08,-5.605880e+08,4.725204e+06,-1.239855e+06,-2.224615e+06
2,2004-07-01 00:30:00.500,2004-07-01 00:40:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,916.660681,0.782019,0.499168,0.638306,...,5.697729e-12,-4.844877e-14,943.898883,2.261063e-14,1.180142e+09,-3.502443e+08,-5.605924e+08,4.683228e+06,-1.389895e+06,-2.224632e+06
3,2004-07-01 00:40:00.500,2004-07-01 00:50:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1091.602839,0.931266,0.499168,0.536010,...,-4.511631e-12,-3.582254e-15,944.036298,-1.790377e-14,1.168368e+09,-3.876862e+08,-5.605966e+08,4.636504e+06,-1.538478e+06,-2.224648e+06
4,2004-07-01 00:50:00.500,2004-07-01 01:00:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1313.677714,1.120722,NaN,NaN,...,3.780981e-12,-1.431667e-13,944.174269,1.500429e-14,1.155410e+09,-4.247230e+08,-5.606005e+08,4.585083e+06,-1.685453e+06,-2.224664e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658981,2017-08-31 22:55:21.500,2017-08-31 23:05:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.626379,0.008212,0.006656,0.810426,...,-3.257067e-13,4.166228e-14,940.335944,-1.292521e-15,1.273147e+09,4.331465e+08,6.761990e+08,5.052303e+06,1.718881e+06,2.683401e+06
658982,2017-08-31 23:05:21.500,2017-08-31 23:15:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.538837,0.008138,0.006656,0.817864,...,2.243471e-12,-6.692418e-15,940.382560,8.902896e-15,1.286398e+09,3.920245e+08,6.761952e+08,5.104891e+06,1.555694e+06,2.683385e+06
658983,2017-08-31 23:15:21.500,2017-08-31 23:25:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.483587,0.008091,0.004992,0.616972,...,-1.586656e-12,5.224539e-14,940.433491,-6.296419e-15,1.298321e+09,3.505024e+08,6.761910e+08,5.152205e+06,1.390919e+06,2.683369e+06
658984,2017-08-31 23:25:21.500,2017-08-31 23:35:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.515213,0.008118,0.006656,0.819895,...,1.395176e-12,5.592884e-16,940.488683,5.536560e-15,1.308903e+09,3.086236e+08,6.761866e+08,5.194198e+06,1.224729e+06,2.683351e+06


# 3D


In [7]:
print(ICW_loc_df)

                interval_start            interval_end  is_ICW  is_mirror  \
0      2004-07-01 00:10:00.500 2004-07-01 00:20:00.500   False      False   
1      2004-07-01 00:20:00.500 2004-07-01 00:30:00.500   False      False   
2      2004-07-01 00:30:00.500 2004-07-01 00:40:00.500   False      False   
3      2004-07-01 00:40:00.500 2004-07-01 00:50:00.500   False      False   
4      2004-07-01 00:50:00.500 2004-07-01 01:00:00.500   False      False   
...                        ...                     ...     ...        ...   
658981 2017-08-31 22:55:21.500 2017-08-31 23:05:21.500   False      False   
658982 2017-08-31 23:05:21.500 2017-08-31 23:15:21.500   False      False   
658983 2017-08-31 23:15:21.500 2017-08-31 23:25:21.500   False      False   
658984 2017-08-31 23:25:21.500 2017-08-31 23:35:21.500   False      False   
658985 2017-08-31 23:35:21.500 2017-08-31 23:45:21.500    True      False   

            label                                          file_name  \
0  

In [8]:
df=ICW_loc_df

In [11]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ============================================================
# Settings
# ============================================================
R_ENCELADUS_KM = 252.1

n_bins = 36
min_count = 10

# Focus region around Enceladus in Enceladus radii.
# This keeps Enceladus and Saturn visible.
# Set to None to use the full available range.
axis_limit_RE = 2500

# ============================================================
# Coordinates: Cassini in Enceladus-centered ENIS frame
# ============================================================
d = df[
    [
        "x_enis_km",
        "y_enis_km",
        "z_enis_km",
        "saturn_x_enis_km",
        "saturn_y_enis_km",
        "saturn_z_enis_km",
        "is_mirror",
    ]
].dropna().copy()

# Convert Cassini position to Enceladus radii
d["x_RE"] = d["x_enis_km"] / R_ENCELADUS_KM
d["y_RE"] = d["y_enis_km"] / R_ENCELADUS_KM
d["z_RE"] = d["z_enis_km"] / R_ENCELADUS_KM

# Convert Saturn position to Enceladus radii
d["saturn_x_RE"] = d["saturn_x_enis_km"] / R_ENCELADUS_KM
d["saturn_y_RE"] = d["saturn_y_enis_km"] / R_ENCELADUS_KM
d["saturn_z_RE"] = d["saturn_z_enis_km"] / R_ENCELADUS_KM

d["is_mirror"] = d["is_mirror"].astype(int)

# Saturn is effectively fixed in this frame; use median for robustness
saturn_pos = np.array(
    [
        d["saturn_x_RE"].median(),
        d["saturn_y_RE"].median(),
        d["saturn_z_RE"].median(),
    ]
)

# ============================================================
# Optional spatial focus around Enceladus
# ============================================================
if axis_limit_RE is not None:
    keep = (
        d["x_RE"].between(-axis_limit_RE, axis_limit_RE)
        & d["y_RE"].between(-axis_limit_RE, axis_limit_RE)
        & d["z_RE"].between(-axis_limit_RE, axis_limit_RE)
    )
    d_plot = d.loc[keep].copy()

    bin_edges = [
        np.linspace(-axis_limit_RE, axis_limit_RE, n_bins + 1),
        np.linspace(-axis_limit_RE, axis_limit_RE, n_bins + 1),
        np.linspace(-axis_limit_RE, axis_limit_RE, n_bins + 1),
    ]
else:
    d_plot = d.copy()

    bin_edges = [
        np.linspace(d_plot["x_RE"].min(), d_plot["x_RE"].max(), n_bins + 1),
        np.linspace(d_plot["y_RE"].min(), d_plot["y_RE"].max(), n_bins + 1),
        np.linspace(d_plot["z_RE"].min(), d_plot["z_RE"].max(), n_bins + 1),
    ]

# ============================================================
# 3D histogram binning
# ============================================================
coords = d_plot[["x_RE", "y_RE", "z_RE"]].to_numpy()
icw = d_plot["is_mirror"].to_numpy()

# Total samples per bin
total_counts, edges = np.histogramdd(
    coords,
    bins=bin_edges
)

# ICW samples per bin
icw_counts, _ = np.histogramdd(
    coords,
    bins=bin_edges,
    weights=icw
)

with np.errstate(divide="ignore", invalid="ignore"):
    occurrence_rate = icw_counts / total_counts

# Bin centers
x_centers = 0.5 * (edges[0][:-1] + edges[0][1:])
y_centers = 0.5 * (edges[1][:-1] + edges[1][1:])
z_centers = 0.5 * (edges[2][:-1] + edges[2][1:])

X, Y, Z = np.meshgrid(
    x_centers,
    y_centers,
    z_centers,
    indexing="ij"
)

# Keep only sufficiently sampled bins
mask = total_counts >= min_count

plot_df = pd.DataFrame(
    {
        "x_RE": X[mask],
        "y_RE": Y[mask],
        "z_RE": Z[mask],
        "n_total": total_counts[mask].astype(int),
        "n_ICW": icw_counts[mask].astype(int),
        "occurrence_rate": occurrence_rate[mask],
    }
)

# Marker size scaled by sampling density
plot_df["marker_size"] = 3 + 9 * np.sqrt(
    plot_df["n_total"] / plot_df["n_total"].max()
)

plot_df["hover"] = (
    "ICW occurrence rate: "
    + plot_df["occurrence_rate"].map("{:.2%}".format)
    + "<br>Total samples: "
    + plot_df["n_total"].astype(str)
    + "<br>ICW samples: "
    + plot_df["n_ICW"].astype(str)
    + "<br>x: "
    + plot_df["x_RE"].map("{:.1f}".format)
    + " R<sub>E</sub>"
    + "<br>y: "
    + plot_df["y_RE"].map("{:.1f}".format)
    + " R<sub>E</sub>"
    + "<br>z: "
    + plot_df["z_RE"].map("{:.1f}".format)
    + " R<sub>E</sub>"
)

# ============================================================
# Optional Saturn sphere
# ============================================================
def make_sphere(center, radius, n_theta=40, n_phi=20):
    theta = np.linspace(0, 2 * np.pi, n_theta)
    phi = np.linspace(0, np.pi, n_phi)

    theta, phi = np.meshgrid(theta, phi)

    xs = center[0] + radius * np.sin(phi) * np.cos(theta)
    ys = center[1] + radius * np.sin(phi) * np.sin(theta)
    zs = center[2] + radius * np.cos(phi)

    return xs, ys, zs


saturn_radius_RE = 60268.0 / R_ENCELADUS_KM
sat_x, sat_y, sat_z = make_sphere(
    center=saturn_pos,
    radius=saturn_radius_RE
)

# ============================================================
# Plotly graph_objects figure
# ============================================================
fig = go.Figure()

# ICW occurrence-rate bins
fig.add_trace(
    go.Scatter3d(
        x=plot_df["x_RE"],
        y=plot_df["y_RE"],
        z=plot_df["z_RE"],
        mode="markers",
        marker=dict(
            size=plot_df["marker_size"],
            color=plot_df["occurrence_rate"],
            colorscale="Viridis",
            cmin=0,
            cmax=1,
            opacity=0.78,
            colorbar=dict(
                title="ICW<br>occurrence<br>rate",
                tickformat=".0%",
            ),
        ),
        text=plot_df["hover"],
        hoverinfo="text",
        name="ICW occurrence rate",
    )
)

# Enceladus at origin
fig.add_trace(
    go.Scatter3d(
        x=[0],
        y=[0],
        z=[0],
        mode="markers+text",
        marker=dict(
            size=8,
            color="black",
        ),
        text=["Enceladus"],
        textposition="top center",
        hoverinfo="text",
        name="Enceladus",
    )
)

# Saturn center
fig.add_trace(
    go.Scatter3d(
        x=[saturn_pos[0]],
        y=[saturn_pos[1]],
        z=[saturn_pos[2]],
        mode="markers+text",
        marker=dict(
            size=7,
            color="orange",
        ),
        text=["Saturn center"],
        textposition="top center",
        hoverinfo="text",
        name="Saturn center",
    )
)

# Saturn physical sphere
fig.add_trace(
    go.Surface(
        x=sat_x,
        y=sat_y,
        z=sat_z,
        opacity=0.18,
        showscale=False,
        colorscale=[[0, "orange"], [1, "orange"]],
        name="Saturn",
        hoverinfo="skip",
    )
)

# Line from Enceladus to Saturn
fig.add_trace(
    go.Scatter3d(
        x=[0, saturn_pos[0]],
        y=[0, saturn_pos[1]],
        z=[0, saturn_pos[2]],
        mode="lines",
        line=dict(
            color="orange",
            width=4,
            dash="dash",
        ),
        name="Enceladus–Saturn direction",
        hoverinfo="skip",
    )
)

# Axis limits
if axis_limit_RE is not None:
    axis_range = [-axis_limit_RE, axis_limit_RE]
else:
    max_abs = np.nanmax(
        np.abs(
            np.concatenate(
                [
                    plot_df[["x_RE", "y_RE", "z_RE"]].to_numpy().ravel(),
                    saturn_pos,
                ]
            )
        )
    )
    max_abs = np.ceil(max_abs / 100) * 100
    axis_range = [-max_abs, max_abs]

fig.update_layout(
    title=dict(
        text="ICW Occurrence Rate in Enceladus-Centered ENIS Coordinates",
        x=0.5,
        xanchor="center",
    ),
    scene=dict(
        xaxis=dict(
            title="X ENIS [R<sub>E</sub>]",
            range=axis_range,
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="gray",
        ),
        yaxis=dict(
            title="Y ENIS [R<sub>E</sub>]",
            range=axis_range,
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="gray",
        ),
        zaxis=dict(
            title="Z ENIS [R<sub>E</sub>]",
            range=axis_range,
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="gray",
        ),
        aspectmode="cube",
    ),
    margin=dict(l=0, r=0, t=70, b=0),
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor="rgba(255,255,255,0.7)",
    ),
)

fig.show()

# save
fig.write_html("mirror_3D_occurrence_ENIS.html")

In [ ]:
ICW_loc_df

,interval_start,interval_end,is_ICW,file_name,time,utc,et,x_enis_km,y_enis_km,z_enis_km,...,saturn_z_enis_km,saturn_x_enis_RE,saturn_y_enis_RE,saturn_z_enis_RE,sun_x_enis_km,sun_y_enis_km,sun_z_enis_km,sun_x_enis_RE,sun_y_enis_RE,sun_z_enis_RE
0,2004-05-14 00:10:24.500,2004-05-14 00:20:24.500,False,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,2004-05-14 00:15:24.500,2004-05-14 00:15:00,1.377658e+08,-2.329855e+07,-6.010075e+06,-7.124432e+06,...,3.421951e-12,943.991689,-1.002338e-14,1.357953e-14,-7.836272e+07,-1.224916e+09,-5.674773e+08,-3.109714e+05,-4.860905e+06,-2.251954e+06
1,2004-05-14 00:20:24.500,2004-05-14 00:30:24.500,False,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,2004-05-14 00:25:24.500,2004-05-14 00:25:00,1.377664e+08,-2.347501e+07,-5.256121e+06,-7.123562e+06,...,-1.776790e-12,944.130227,-1.192336e-14,-7.050940e-15,-1.173608e+08,-1.221787e+09,-5.674816e+08,-4.657297e+05,-4.848488e+06,-2.251971e+06
2,2004-05-14 00:30:24.500,2004-05-14 00:40:24.500,False,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,2004-05-14 00:35:24.500,2004-05-14 00:35:00,1.377670e+08,-2.362729e+07,-4.497248e+06,-7.122686e+06,...,2.305288e-12,944.269236,-6.363557e-15,9.148209e-15,-1.562280e+08,-1.217418e+09,-5.674858e+08,-6.199687e+05,-4.831150e+06,-2.251988e+06
3,2004-05-14 00:40:24.500,2004-05-14 00:50:24.500,False,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,2004-05-14 00:45:24.500,2004-05-14 00:45:00,1.377676e+08,-2.375526e+07,-3.734231e+06,-7.121804e+06,...,2.954970e-12,944.408574,-6.020744e-15,1.172638e-14,-1.949249e+08,-1.211814e+09,-5.674902e+08,-7.735321e+05,-4.808914e+06,-2.252005e+06
4,2004-05-14 00:50:24.500,2004-05-14 01:00:24.500,False,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,2004-05-14 00:55:24.500,2004-05-14 00:55:00,1.377682e+08,-2.385882e+07,-2.967851e+06,-7.120916e+06,...,2.813536e-12,944.548100,1.303487e-15,1.116512e-14,-2.334125e+08,-1.204983e+09,-5.674946e+08,-9.262646e+05,-4.781807e+06,-2.252022e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665804,2017-08-31 22:55:21.500,2017-08-31 23:05:21.500,False,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,2017-08-31 23:00:21.500,2017-08-31 23:00:00,5.574925e+08,-7.574319e+04,9.538960e+05,3.139608e+05,...,-3.257067e-13,940.335944,-5.380997e-15,-1.292521e-15,4.331465e+08,-1.273147e+09,6.761990e+08,1.718881e+06,-5.052303e+06,2.683401e+06
665805,2017-08-31 23:05:21.500,2017-08-31 23:15:21.500,False,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,2017-08-31 23:10:21.500,2017-08-31 23:10:00,5.574931e+08,-4.486749e+04,9.611225e+05,3.143557e+05,...,2.243471e-12,940.382560,7.446328e-15,8.902896e-15,3.920245e+08,-1.286398e+09,6.761952e+08,1.555694e+06,-5.104891e+06,2.683385e+06
665806,2017-08-31 23:15:21.500,2017-08-31 23:25:21.500,False,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,2017-08-31 23:20:21.500,2017-08-31 23:20:00,5.574937e+08,-1.384946e+04,9.673408e+05,3.147472e+05,...,2.323338e-13,940.433491,3.437544e-14,9.219839e-16,3.505024e+08,-1.298321e+09,6.761910e+08,1.390919e+06,-5.152205e+06,2.683369e+06
665807,2017-08-31 23:25:21.500,2017-08-31 23:35:21.500,False,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,2017-08-31 23:30:21.500,2017-08-31 23:30:00,5.574943e+08,1.727742e+04,9.725492e+05,3.151350e+05,...,1.395176e-12,940.488683,-5.592884e-16,5.536560e-15,3.086236e+08,-1.308903e+09,6.761866e+08,1.224729e+06,-5.194198e+06,2.683351e+06
